# RISS Inference Demo

This notebook provides a lightweight example for running inference with pretrained RISS weights. It is intended to demonstrate the workflow:

1. check the environment;
2. locate pretrained weights;
3. prepare an example low-energy CEM or mammography input;
4. run inference with the repository `test.py` entry point;
5. visualize and save the synthetic recombined image.

This notebook is **not** intended to reproduce all manuscript-level quantitative results. Full evaluation requires the complete test cohorts, standardized preprocessing, and the evaluation scripts described in the README.

## 1. Environment Check

Run this cell first to confirm that PyTorch and CUDA are available. CPU inference may work for debugging but is usually slow.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import torch
from PIL import Image
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

## 2. Set Repository Paths

The notebook assumes the following repository layout:

```text
RISS/
  test.py
  Checkpoints/le_re/latest_net_G.pth
  notebooks/RISS_inference_demo.ipynb
  assets/example/sample_LE.png
```

If your checkpoint is stored elsewhere, update `checkpoint_dir` or `experiment_name` below.

In [ ]:
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

test_py = repo_root / "test.py"
checkpoint_dir = repo_root / "Checkpoints"
experiment_name = "le_re"
weights_path = checkpoint_dir / experiment_name / "latest_net_G.pth"

example_le = repo_root / "notebooks" / "assets" / "example" / "sample_LE.png"
example_mammography = repo_root / "notebooks" / "assets" / "example" / "sample_mammography.png"
demo_dataset = repo_root / "Datasets" / "notebook_demo"
results_dir = repo_root / "results" / "notebook_demo"

print("Repository root:", repo_root)
print("test.py exists:", test_py.exists(), test_py)
print("Weights path:", weights_path)
print("Weights exist:", weights_path.exists())

## 3. Prepare a Single-Image Demo Input

RISS uses the aligned image format from the original training pipeline: each PNG is a horizontal A|B concatenation where the **left half is the low-energy input** and the **right half is the recombined target** (a placeholder is fine at inference time).

The bundled `sample_LE.png` is already stored in this aligned format, so we simply **copy it into the dataset folder as-is**. We deliberately do **not** convert it to grayscale, resize it, or zero out channels: the model was trained on the original three-channel aligned images, and altering the input would push it off the training distribution and degrade the synthetic output.

For your own data, place preprocessed aligned images (left = LE, right = target/placeholder) directly under `Datasets/<name>/test/`. If you only have a raw single-channel LE image, first build an aligned pair by concatenating it with a placeholder side by side.

In [ ]:
import shutil

# The example image is already in the aligned A|B format expected by the
# dataloader (left half = low-energy input, right half = target/placeholder).
# We copy it as-is: no grayscale / resize / channel rewriting is applied,
# otherwise the input would no longer match the training distribution and
# the synthetic output would be degraded.
demo_input_dir = demo_dataset / "test"
demo_input_dir.mkdir(parents=True, exist_ok=True)
demo_input = demo_input_dir / "sample_LE_input.png"
shutil.copy(example_le, demo_input)
print("Demo input copied to:", demo_input)

img = Image.open(demo_input)
w, h = img.size
fig, axes = plt.subplots(1, 2, figsize=(8, 5))
axes[0].imshow(img.crop((0, 0, w // 2, h)))
axes[0].set_title("Input LE (left half)")
axes[0].axis("off")
axes[1].imshow(img.crop((w // 2, 0, w, h)))
axes[1].set_title("Target / placeholder (right half)")
axes[1].axis("off")
plt.tight_layout()

## 4. Run RISS Inference

The recommended path is to call the repository `test.py` entry point, because it reuses the same model creation and preprocessing logic as the command-line workflow.

If the cell reports that `test.py` or weights are missing, download the pretrained weights and place them at:

```text
Checkpoints/le_re/latest_net_G.pth
```

In [ ]:
cmd = [
    sys.executable, str(test_py),
    "--dataroot", str(demo_dataset),
    "--name", experiment_name,
    "--gpu_ids", "0" if torch.cuda.is_available() else "-1",
    "--model", "resvit_one",
    "--which_model_netG", "resvit",
    "--pre_trained_transformer", "0",
    "--dataset_mode", "aligned",
    "--norm", "batch",
    "--phase", "test",
    "--output_nc", "1",
    "--input_nc", "3",
    "--how_many", "1",
    "--serial_batches",
    "--fineSize", "256",
    "--loadSize", "256",
    "--results_dir", str(results_dir),
    "--checkpoints_dir", str(checkpoint_dir),
    "--which_epoch", "latest",
]

if not test_py.exists():
    raise FileNotFoundError(f"Cannot find test.py at {test_py}. Run this notebook from the RISS repository.")
if not weights_path.exists():
    raise FileNotFoundError(f"Cannot find pretrained weights at {weights_path}. Please download latest_net_G.pth first.")

print("Running command:")
print(" ".join(cmd))
completed = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(f"Inference failed with exit code {completed.returncode}")

## 5. Find and Visualize the Output

Different versions of the image-to-image translation template may save generated images under slightly different subdirectories. This cell searches the demo result directory for likely synthetic recombined images.

In [ ]:
image_exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
candidate_outputs = [
    p for p in results_dir.rglob("*")
    if p.suffix.lower() in image_exts and ("fake" in p.name.lower() or "syn" in p.name.lower() or "output" in p.name.lower())
]
if not candidate_outputs:
    candidate_outputs = [p for p in results_dir.rglob("*") if p.suffix.lower() in image_exts]

print("Found output images:")
for p in candidate_outputs[:10]:
    print(" -", p)

if not candidate_outputs:
    raise FileNotFoundError(f"No output image found under {results_dir}")

output_image = candidate_outputs[0]
fig, axes = plt.subplots(1, 2, figsize=(8, 5))
axes[0].imshow(Image.open(example_le), cmap="gray")
axes[0].set_title("Input LE")
axes[0].axis("off")
axes[1].imshow(Image.open(output_image), cmap="gray")
axes[1].set_title("Synthetic recombined image")
axes[1].axis("off")
plt.tight_layout()
print("Visualized output:", output_image)

## 6. Batch Inference Equivalent

For full test-set inference, use the command-line workflow below and replace `--dataroot` with your preprocessed dataset directory.

In [ ]:
print(f'''
python test.py \\
  --dataroot Datasets/your_dataset \\
  --name {experiment_name} \\
  --gpu_ids 0 \\
  --model resvit_one \\
  --which_model_netG resvit \\
  --dataset_mode aligned \\
  --norm batch \\
  --phase test \\
  --output_nc 1 \\
  --input_nc 3 \\
  --how_many 10000 \\
  --serial_batches \\
  --fineSize 256 \\
  --loadSize 256 \\
  --results_dir results/full_test \\
  --checkpoints_dir Checkpoints \\
  --which_epoch latest
''')

## Notes on Quantitative Results

This notebook is a minimal inference demonstration using example images. Quantitative values obtained from this demo, if any, may differ from manuscript-level results because the paper reports aggregate performance over full test cohorts with standardized preprocessing and statistical analysis.

To reproduce manuscript-level metrics, run inference on the complete test set and then use the evaluation scripts for PSNR, SSIM, and downstream analyses.